In [0]:
# %load_ext autoreload
# %autoreload 2
# %reload_ext autoreload

In [0]:
%run ../../config/utils

In [0]:
"""
The script which merges all the intermediate files and writes the updated cube
"""

import sys
sys.path.append('..')
sys.path.append('../..')

import lib_dna_member.generate_population as gp
import lib_dna_member.job_manager as job_manager
from lib_dna_member.s3 import member_dna_input_data_validator
from databricks.feature_engineering import FeatureEngineeringClient
from datetime import datetime
import pyspark.sql.functions as sqlf

In [0]:
dbutils.widgets.text("run_as_date", "", "Date for data processing") # create the widget if missing

date_of_run_str = dbutils.widgets.get("run_as_date")
run_as_date = datetime.strptime(date_of_run_str, "%Y-%m-%d").date() 

print(f"Run as date:    {run_as_date}")

In [0]:
def merge(job):
    """
    Merge all the modules for the given population

    Parameters:
        job (managers.JobManager): object which manages the Spark App
    Returns:
        (pyspark.sql.DataFrame): dna with all the features
    """
    print("IN MERGE SCRIPT ")

    dna = job.tables["population"]
    dna = dna.repartition("MBRSHP_SID", "FISCAL_WEEK_END")

    intermediates = [
        "cubes_transaction_1",
        "cubes_transaction_2",
        "cubes_transaction_3",
        "cubes_coupon_and_digital_1",
        "cubes_coupon_and_digital_2",
        "cubes_member_features",
        "cubes_most_shopped",
        "cubes_misc",
        "cubes_acquisition",
    ]

    for i in intermediates:
        features = job.tables[i].repartition( "MBRSHP_SID", "FISCAL_WEEK_END" )
        dna = dna.join(
            features,
            ["MBRSHP_SID", "FISCAL_WEEK_END"],
            "left_outer",
        )

    return dna

In [0]:
job = job_manager.JobManager(spark, intermediate_all_tables_dict, member_dna_config_path)

In [0]:
recency_lookback_duration = job.config["params"].get(
    "recency_lookback_duration", {}
)

member_dna_input_data_validator(
    silver_skeleton, 
    silver_master_member_extended, 
    fs_cubes_transaction_1, fs_cubes_transaction_2, fs_cubes_transaction_3,
    fs_cubes_coupon_and_digital_1, fs_cubes_coupon_and_digital_2, 
    fs_cubes_member, 
    fs_cubes_most_shopped, 
    fs_cubes_misc, 
    fs_cubes_acquisition,
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)

In [0]:
job.read_table("skeleton") 
job.read_table("member_extended")
job.read_table("cubes_transaction_1")
job.read_table("cubes_transaction_2")
job.read_table("cubes_transaction_3")
job.read_table("cubes_coupon_and_digital_1")
job.read_table("cubes_coupon_and_digital_2")
job.read_table("cubes_member_features")
job.read_table("cubes_most_shopped")
job.read_table("cubes_misc")
job.read_table("cubes_acquisition")

In [0]:
population = gp.generate_population(job)
job.tables["population"] = population

dna_features = merge(job)

dna_features = dna_features.filter(
    dna_features.FISCAL_WEEK_END >= job.config["params"]["start_date"]
)

### Save results

In [0]:
#### IMPORTANT     This was NOT in the original code but the target table has a NOT NULL constrain on this column

dna_features = dna_features.filter("MBRSHP_SID IS NOT NULL")

In [0]:
spark.sql(f"DELETE FROM {fs_customer_cube_full}")

fe = FeatureEngineeringClient()

# partitionby="FISCAL_WEEK_END",

fe.write_table(
    name=fs_customer_cube_full,  
    df=dna_features,
    mode="merge"
)
# this was originally saved to "s3://memberanalytics-data-out-prod/CUBES/customer_cube_full/customer_cube"

In [0]:
if job.config["params"]["archive"]:
    max_row = spark.table(customer_cube_archive).agg(sqlf.max("FISCAL_WEEK_END")).first()
    max_date = max_row[0] if max_row else None

    if max_date is not None:
        dna_features = dna_features.filter(sqlf.col("FISCAL_WEEK_END") > sqlf.lit(max_date))
    else:
        pass 

    save_archive(dna_features, customer_cube_archive, run_as_date)

In [0]:
# We will save to DBX Volume by default. If in prod we additionally move such file to s3
dna_to_s3 = spark.table(fs_customer_cube_full)
dna_to_s3.repartition("FISCAL_WEEK_END").write.mode('overwrite').partitionBy('FISCAL_WEEK_END').parquet(dna_cube_volume_path)

print(f"Output successfully saved to volume: {dna_cube_volume_path}.")

# LL Note: we recommend EDW team to access this directly from UnityCatalog and delete this step and their related files/ resources

if environment == 'prod':
    try:
        dna_to_s3.repartition("FISCAL_WEEK_END").write.mode('overwrite').partitionBy('FISCAL_WEEK_END').parquet(dna_cube_edw_path)
        print(f"Inference output saved to additional S3 locations ({dna_cube_edw_path}) successfully.")
    except Exception as e:
        print(f"Error saving results to S3: {e}")  